# Notebook 2 - Chunking Strategies for RAG
## Fixed vs Recursive vs Semantic Chunking


This notebook demonstrates how different chunking strategies affect retrieval quality in a RAG system using Ollama + Nomic embeddings.


In [32]:
pip_packages = [
    'langchain',
    'langchain-community',
    'langchain-text-splitters',
    'langchain-experimental',
    'langchain-ollama',
    'chromadb'
]
print('Install if needed:')
print('pip install ' + ' '.join(pip_packages))

Install if needed:
pip install langchain langchain-community langchain-text-splitters langchain-experimental langchain-ollama chromadb


In [33]:
from langchain_core.documents import Document

text = '''
Company Policy Document

Maternity Leave Policy

Employees are eligible for 6 months maternity leave.
Maternity leave can be extended under special circumstances.
Medical certificates may be required.
HR approval is mandatory.

Resignation Policy

Employees must serve 60 days notice period before resignation.
Knowledge transfer activities must be completed.
Company assets must be returned before final settlement.
Manager approval is required.

Annual Leave Policy

Annual leave balance is 24 days.
Unused leaves may be carried forward.
Leave requests should be submitted in advance.
Managers may reject leave requests during critical projects.

Work From Home Policy

Work from home is allowed twice a week.
Manager approval may be required.
Repeated policy violations may lead to revocation.
Employees must be available during business hours.
'''

documents = [Document(page_content=text)]
print('Document Loaded')

Document Loaded


In [34]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separator=' '
)

fixed_chunks = fixed_splitter.split_documents(documents)
print('Fixed Chunks:', len(fixed_chunks))

for i,c in enumerate(fixed_chunks):
    print(f'\nChunk {i+1}\n')
    print(c.page_content)


Fixed Chunks: 6

Chunk 1

Company Policy Document

Maternity Leave Policy

Employees are eligible for 6 months maternity leave.
Maternity leave can be extended under special

Chunk 2

circumstances.
Medical certificates may be required.
HR approval is mandatory.

Resignation Policy

Employees must serve 60 days notice period before

Chunk 3

resignation.
Knowledge transfer activities must be completed.
Company assets must be returned before final settlement.
Manager approval is

Chunk 4

required.

Annual Leave Policy

Annual leave balance is 24 days.
Unused leaves may be carried forward.
Leave requests should be submitted in

Chunk 5

advance.
Managers may reject leave requests during critical projects.

Work From Home Policy

Work from home is allowed twice a week.
Manager approval

Chunk 6

may be required.
Repeated policy violations may lead to revocation.
Employees must be available during business hours.


In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0
)

recursive_chunks = recursive_splitter.split_documents(documents)
print('Recursive Chunks:', len(recursive_chunks))

for i,c in enumerate(recursive_chunks):
    print(f'\nChunk {i+1}\n')
    print(c.page_content)


Recursive Chunks: 12

Chunk 1

Company Policy Document

Maternity Leave Policy

Chunk 2

Employees are eligible for 6 months maternity leave.
Maternity leave can be extended under special circumstances.

Chunk 3

Medical certificates may be required.
HR approval is mandatory.

Chunk 4

Resignation Policy

Chunk 5

Employees must serve 60 days notice period before resignation.
Knowledge transfer activities must be completed.

Chunk 6

Company assets must be returned before final settlement.
Manager approval is required.

Chunk 7

Annual Leave Policy

Chunk 8

Annual leave balance is 24 days.
Unused leaves may be carried forward.
Leave requests should be submitted in advance.

Chunk 9

Managers may reject leave requests during critical projects.

Chunk 10

Work From Home Policy

Chunk 11

Work from home is allowed twice a week.
Manager approval may be required.
Repeated policy violations may lead to revocation.

Chunk 12

Employees must be available during business hours.


In [36]:
!pip install langchain_experimental

In [44]:
from langchain_ollama import OllamaEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = OllamaEmbeddings(model='nomic-embed-text')

#semantic_splitter = SemanticChunker(embeddings)
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=25
)
semantic_chunks = semantic_splitter.split_documents(documents)

print('Semantic Chunks:', len(semantic_chunks))

for i,c in enumerate(semantic_chunks):
    print(f'\nChunk {i+1}\n')
    print(c.page_content)


Semantic Chunks: 13

Chunk 1


Company Policy Document

Maternity Leave Policy

Employees are eligible for 6 months maternity leave. Maternity leave can be extended under special circumstances.

Chunk 2

Medical certificates may be required.

Chunk 3

HR approval is mandatory.

Chunk 4

Resignation Policy

Employees must serve 60 days notice period before resignation.

Chunk 5

Knowledge transfer activities must be completed.

Chunk 6

Company assets must be returned before final settlement.

Chunk 7

Manager approval is required.

Chunk 8

Annual Leave Policy

Annual leave balance is 24 days. Unused leaves may be carried forward.

Chunk 9

Leave requests should be submitted in advance.

Chunk 10

Managers may reject leave requests during critical projects. Work From Home Policy

Work from home is allowed twice a week.

Chunk 11

Manager approval may be required.

Chunk 12

Repeated policy violations may lead to revocation. Employees must be available during business hours.

Chunk 13



In [38]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='nomic-embed-text')

fixed_db = Chroma.from_documents(fixed_chunks, embeddings)
recursive_db = Chroma.from_documents(recursive_chunks, embeddings)
semantic_db = Chroma.from_documents(semantic_chunks, embeddings)


In [43]:
print(len(documents[0].page_content))
print(len(fixed_chunks))
print(len(recursive_chunks))

849
6
12


In [39]:
question = 'How many work from home days are allowed per month?'

print('QUESTION:', question)


QUESTION: How many work from home days are allowed per month?


In [40]:
print('\n===== FIXED CHUNKING =====')
results = fixed_db.similarity_search(question, k=2)
for r in results:
    print(r.page_content)



===== FIXED CHUNKING =====
Work From Home Policy
Employees may work from home up to 8 days per month.
Manager approval is required.
Employees may work from home up to 8 days per month.
Manager approval is required.
Repeated policy violations may result in revocation.
Travel Policy


In [41]:
print('\n===== RECURSIVE CHUNKING =====')
results = recursive_db.similarity_search(question, k=2)
for r in results:
    print(r.page_content)



===== RECURSIVE CHUNKING =====
Work From Home Policy
Employees may work from home up to 8 days per month.
Manager approval is required.
Employees may work from home up to 8 days per month.
Manager approval is required.
Repeated policy violations may result in revocation.
Travel Policy


In [42]:
print('\n===== SEMANTIC CHUNKING =====')
results = semantic_db.similarity_search(question, k=2)
for r in results:
    print(r.page_content)



===== SEMANTIC CHUNKING =====
Work From Home Policy
Employees may work from home up to 8 days per month.
Manager approval is required.
Employees may work from home up to 8 days per month.
Manager approval is required.
Repeated policy violations may result in revocation.
Travel Policy


## Expected Learning

- Fixed chunking may split related information.
- Recursive chunking usually preserves structure better.
- Semantic chunking groups information by meaning.
- Better chunks generally lead to better retrieval quality.
